# AutoML for Time Series — sktime
## Data Loading

## Packages

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.compose import RecursiveTabularRegressionForecaster
from sktime.forecasting.fbprophet import Prophet
from sktime.forecasting.arima import ARIMA as SktimeSARIMA
from xgboost import XGBRegressor

Importing plotly failed. Interactive plots will not work.


In [ ]:
data_dir = os.path.dirname(os.path.abspath("sktime.ipynb"))

# --- Raw files ---
raw_files = sorted(glob.glob(os.path.join(data_dir, "EF* - OD.csv")))

dfs_raw = {}
for path in raw_files:
    station = os.path.basename(path).split(" - ")[0]
    df = pd.read_csv(path, sep=";", decimal=",", parse_dates=["Data"], dayfirst=True)
    df = df.rename(columns={"Data": "timestamp"})
    df["station"] = station
    dfs_raw[station] = df
    print(f"{station} raw: {df.shape}  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

print()

# --- Complete series files ---
complete_series_files = sorted(glob.glob(os.path.join(data_dir, "*_DO_complete_series.csv")))

dfs_complete_series = {}
for path in complete_series_files:
    station = os.path.basename(path).split("_")[0]
    df = pd.read_csv(path, sep=";", decimal=",", parse_dates=["Data"])
    df = df.rename(columns={"Data": "timestamp", "DO_filled_mgL": "DO_mgL"})
    df["station"] = station
    dfs_complete_series[station] = df
    print(f"{station} complete_series: {df.shape}  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

# --- Fallback: imputed files (for stations without complete_series, e.g. EF03) ---
imputed_files = sorted(glob.glob(os.path.join(data_dir, "*_DO_imputed.csv")))
for path in imputed_files:
    station = os.path.basename(path).split("_")[0]
    if station in dfs_complete_series:
        continue  # already has complete_series
    df = pd.read_csv(path, sep=";", decimal=",", parse_dates=["Data"])
    df = df.rename(columns={"Data": "timestamp", "DO_imputed_mgL": "DO_mgL"})
    df["station"] = station
    dfs_complete_series[station] = df
    print(f"{station} imputed (fallback): {df.shape}  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

## Data Preparation for sktime

### Constants and Temporal Split

In [ ]:
START_DATE        = "2015-01-01"
CUTOFF            = pd.Timestamp("2025-01-01")   # start of test period
CUTOFF_END        = pd.Timestamp("2025-04-01")   # end of test period (exclusive)
PREDICTION_LENGTH = int((CUTOFF_END - CUTOFF).total_seconds() // 3600)

print(f"Test period      : {CUTOFF.date()} → {(CUTOFF_END - pd.Timedelta(hours=1)).date()}")
print(f"Prediction length: {PREDICTION_LENGTH} h  ({PREDICTION_LENGTH // 24} days)")

### Hourly Series: Raw and Complete Series Versions

In [ ]:
series_raw             = {}
series_complete_series = {}

for station in sorted(dfs_raw):
    if station not in dfs_complete_series:
        print(f"{station}: no imputed data available, skipping.")
        continue

    # ── RAW ──────────────────────────────────────────────────────────────
    raw = dfs_raw[station][["timestamp", "Media"]].copy()
    raw["DO_mgL"] = pd.to_numeric(raw["Media"], errors="coerce")
    raw = raw[["timestamp", "DO_mgL"]].set_index("timestamp")

    # ── COMPLETE SERIES ───────────────────────────────────────────────────
    imp = dfs_complete_series[station].set_index("timestamp")[["DO_mgL"]].copy()

    # ── Common window (ensures same start and end) ─────────────────────
    common_start = max(raw.index.min(), imp.index.min(), pd.Timestamp(START_DATE))
    common_end   = min(raw.index.max(), imp.index.max())

    raw = raw[(raw.index >= common_start) & (raw.index <= common_end)]
    imp = imp[(imp.index >= common_start) & (imp.index <= common_end)]

    # ── Full hourly grid ────────────────────────────────────────────────
    full_idx = pd.date_range(common_start, common_end, freq="h")
    raw = raw.reindex(full_idx); raw.index.name = "timestamp"
    imp = imp.reindex(full_idx); imp.index.name = "timestamp"

    series_raw[station]             = raw["DO_mgL"].asfreq("h")
    series_complete_series[station] = imp["DO_mgL"].asfreq("h")

    print(f"{station}  window: {common_start.date()} → {common_end.date()}")
    print(f"  raw             : {len(raw):,} pts | NaN: {raw['DO_mgL'].isna().sum():,}")
    print(f"  complete_series : {len(imp):,} pts | NaN: {imp['DO_mgL'].isna().sum():,}")
    print()

In [ ]:
# ── Baseline I  — ffill().bfill() ────────────────────────────────────────────
# Propagates the last observed value forward (and the first backward for leading NaN).
# Fast and parameter-free, but does not respect local trends.
# Filling restricted to training period (< CUTOFF) to avoid data leakage.
series_baseline_I = {}
for station, raw_s in series_raw.items():
    b1 = raw_s.copy()
    mask = b1.index < CUTOFF
    b1.loc[mask] = b1.loc[mask].ffill().bfill()
    series_baseline_I[station] = b1
    print(f"{station}  Baseline I  — training NaN: {b1.loc[mask].isna().sum():,}")

print()

# ── Baseline II — linear interpolation ───────────────────────────────────────
# Fills internal gaps with linear interpolation between the two nearest observed
# neighbours; ffill/bfill only at boundaries without prior/posterior neighbours.
# Respects local trend but assumes linearity between known points.
# Filling restricted to training period (< CUTOFF) to avoid data leakage.
series_baseline_II = {}
for station, raw_s in series_raw.items():
    b2 = raw_s.copy()
    mask = b2.index < CUTOFF
    b2.loc[mask] = (
        b2.loc[mask]
        .interpolate(method="linear")
        .ffill()
        .bfill()
    )
    series_baseline_II[station] = b2
    print(f"{station}  Baseline II — training NaN: {b2.loc[mask].isna().sum():,}")

### Differences Between Baselines

| | **Baseline I** (ffill/bfill) | **Baseline II** (linear interpolation) | **Hierarchical Filling** |
|---|---|---|---|
| **Method** | Copies the last observed value forward; the first backward for leading NaN | Linearly interpolates between the two nearest observed neighbours of each gap | Hierarchical pipeline: Interp. (≤10 h) → MA (≤42 h) → STL (≤72 h) → Prophet (> 72 h) |
| **Local trend** | No — holds the value constant throughout the gap | Yes — linear transition between neighbouring points | Yes — captures daily/weekly/annual seasonality and long-term trend |
| **Long gaps** | May propagate a value for days or weeks | May produce unrealistic ramps over very long gaps | Uses STL/Prophet which model seasonal patterns |
| **QC / constancy** | No — uses raw series with original NaN | No — uses raw series with original NaN | Yes — removes physical outliers and flatline periods before imputation |
| **Role here** | Lower reference bound — trivial strategy | Upper reference bound — classical strategy | Treated series under evaluation |
| **Anti-leakage** | Filling restricted to `index < CUTOFF`; test evaluated against original raw series | Same | External file generated before the test period |

In [ ]:
# ── Save baseline series to CSV for use in other notebooks ────────────────────
# Filling is applied to the FULL series (train + test) here, as these CSVs
# are intended for other contexts that will enforce their own leakage control.
# Format compatible with *_DO_complete_series.csv files from the pipeline.

for station, raw_s in series_raw.items():
    full = raw_s.copy()

    # Baseline I: ffill/bfill global
    bI = full.ffill().bfill()
    dfI = pd.DataFrame({"Data": bI.index, "DO_filled_mgL": bI.values})
    pathI = os.path.join(data_dir, f"{station}_DO_baseline_I.csv")
    dfI.to_csv(pathI, sep=";", decimal=",", index=False)

    # Baseline II: global linear interpolation
    bII = full.interpolate(method="linear").ffill().bfill()
    dfII = pd.DataFrame({"Data": bII.index, "DO_filled_mgL": bII.values})
    pathII = os.path.join(data_dir, f"{station}_DO_baseline_II.csv")
    dfII.to_csv(pathII, sep=";", decimal=",", index=False)

    print(f"{station}  → {os.path.basename(pathI)}  (NaN: {bI.isna().sum()})  |  "
          f"{os.path.basename(pathII)}  (NaN: {bII.isna().sum()})")

print("\nCSVs saved. Columns: Data, DO_filled_mgL")

### Train/Test Split and ForecastingHorizon

In [ ]:
DATASETS = {
    "baseline_I" : series_baseline_I,
    "baseline_II": series_baseline_II,
    "complete_series": series_complete_series,
}

train_series = {ds: {} for ds in DATASETS}
test_series  = {ds: {} for ds in DATASETS}

for ds_name, series_dict in DATASETS.items():
    for station, y in series_dict.items():
        train_series[ds_name][station] = y[y.index < CUTOFF]
        test_series[ds_name][station]  = y[(y.index >= CUTOFF) & (y.index < CUTOFF_END)]

# Raw ground truth — kept separate from DATASETS for fair evaluation
train_raw_sk = {s: y[y.index < CUTOFF]
                for s, y in series_raw.items()}
test_raw_sk  = {s: y[(y.index >= CUTOFF) & (y.index < CUTOFF_END)]
                for s, y in series_raw.items()}

# Absolute forecasting horizon (same for all datasets)
fh = ForecastingHorizon(
    pd.date_range(CUTOFF, CUTOFF_END - pd.Timedelta(hours=1), freq="h"),
    is_relative=False,
)
print(f"Forecasting horizon: {len(fh):,} steps ({len(fh) // 24} days)")

for station in sorted(series_raw):
    y_tr = train_raw_sk[station]
    y_te = test_raw_sk[station]
    print(f"  {station}: train {y_tr.index[0].date()} → {y_tr.index[-1].date()}  |  "
          f"test {y_te.index[0].date()} → {y_te.index[-1].date()}  ({len(y_te):,} h)")

### Train and Test Series Visualization (Raw)

In [ ]:
fig, axes = plt.subplots(len(series_raw), 1, figsize=(14, 4 * len(series_raw)), sharex=False)
if len(series_raw) == 1:
    axes = [axes]

for ax, station in zip(axes, sorted(series_raw)):
    y_train = train_raw_sk[station]
    y_test  = test_raw_sk[station]
    ax.plot(y_train.tail(24 * 90).index, y_train.tail(24 * 90).values,
            color="steelblue", lw=1, label="Train (last 90 days)")
    ax.plot(y_test.index, y_test.values, color="tomato", lw=1, label="Test (raw)")
    ax.set_title(f"{station} — DO (mg/L)")
    ax.set_ylabel("DO (mg/L)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Train / Test Split — Raw Series (Ground Truth)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Modelling — Prophet · XGBoost · SARIMA · DeepAR

In [ ]:
# ── Seasonally scaled metrics (m = 24 h — daily seasonality) ─────────────────
METRIC_M = 24

def _scale_mae(y_train, m=METRIC_M):
    y = np.asarray(y_train, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) <= m:
        return np.nan
    return np.mean(np.abs(y[m:] - y[:-m]))

def _scale_rmse(y_train, m=METRIC_M):
    y = np.asarray(y_train, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) <= m:
        return np.nan
    return np.sqrt(np.mean((y[m:] - y[:-m]) ** 2))

def mase(obs, pred, y_train, m=METRIC_M):
    s = _scale_mae(y_train, m)
    if not s or np.isnan(s):
        return np.nan
    return np.mean(np.abs(obs - pred)) / s

def rmsse(obs, pred, y_train, m=METRIC_M):
    s = _scale_rmse(y_train, m)
    if not s or np.isnan(s):
        return np.nan
    return np.sqrt(np.mean((obs - pred) ** 2)) / s

def r2_score(obs, pred):
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    if ss_tot == 0:
        return np.nan
    return 1 - np.sum((obs - pred) ** 2) / ss_tot

def da(obs, pred):
    """Directional Accuracy — fraction with correct direction (non-flat obs only)."""
    d_obs, d_pred = np.diff(obs), np.diff(pred)
    mask = d_obs != 0
    if not mask.any():
        return np.nan
    return float(np.mean(np.sign(d_obs[mask]) == np.sign(d_pred[mask])))

def mda(obs, pred):
    """Mean Directional Accuracy — all steps including flat."""
    d_obs, d_pred = np.diff(obs), np.diff(pred)
    if len(d_obs) == 0:
        return np.nan
    return float(np.mean(np.sign(d_obs) == np.sign(d_pred)))

def maed(obs, pred):
    """Mean Absolute Error on First Differences."""
    if len(obs) < 2:
        return np.nan
    return float(np.mean(np.abs(np.diff(obs) - np.diff(pred))))

def ce_trend(obs, pred, n_bins=3, eps=1e-9):
    """Cross-Entropy on Quantized Trends (down / flat / up)."""
    d_obs, d_pred = np.diff(obs), np.diff(pred)
    if len(d_obs) == 0:
        return np.nan
    nz = d_obs[d_obs != 0]
    lim = float(np.percentile(np.abs(nz), 90)) if len(nz) > 0 else 1e-6
    edges = np.array([-np.inf, -lim, lim, np.inf])
    p = np.histogram(d_obs,  bins=edges)[0] / len(d_obs)
    q = np.histogram(d_pred, bins=edges)[0] / len(d_pred)
    q = np.clip(q, eps, 1); q /= q.sum()
    return float(-np.sum(p * np.log(q)))

In [ ]:
import time

# Recent training window for SARIMA (infeasible on 87k hourly points)
SARIMA_WINDOW = 24 * 60   # last 60 days

MODELS_CFG = {
    "Prophet": lambda: Prophet(
        seasonality_mode="additive",
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=True,
    ),
    "XGBoost": lambda: RecursiveTabularRegressionForecaster(
        XGBRegressor(n_estimators=300, learning_rate=0.05, n_jobs=-1, random_state=42),
        window_length=24 * 7,
    ),
    "SARIMA": lambda: SktimeSARIMA(
        order=(1, 1, 1),
        seasonal_order=(0, 1, 1, 24),
        suppress_warnings=True,
    ),
}

forecasts   = {ds: {} for ds in DATASETS}
metric_rows = []

for ds_name, series_dict in DATASETS.items():
    print(f"\n{'─'*60}")
    print(f"  Dataset: {ds_name}")
    print(f"{'─'*60}")

    for model_name, model_factory in MODELS_CFG.items():
        print(f"\n  ── {model_name} ──")
        forecasts[ds_name][model_name] = {}

        for station in sorted(series_dict):
            y_train = train_series[ds_name][station]
            y_test  = test_raw_sk[station]      # ground truth always = original raw

            y_train_fit = y_train.ffill().bfill()

            if model_name == "SARIMA":
                y_train_fit = y_train_fit.iloc[-SARIMA_WINDOW:]

            try:
                model = model_factory()
                t0_fit = time.time()
                model.fit(y_train_fit, fh=fh)
                fit_t = time.time() - t0_fit
                t0_pred = time.time()
                pred = model.predict(fh=fh)
                pred_t = time.time() - t0_pred
                forecasts[ds_name][model_name][station] = pred

                mask = y_test.notna() & pred.notna()
                obs_v, pred_v = y_test[mask].values, pred[mask].values
                if len(obs_v) == 0:
                    continue

                y_tr_v = y_train.values

                mase_v    = mase(obs_v, pred_v, y_tr_v)
                rmsse_v   = rmsse(obs_v, pred_v, y_tr_v)
                r2_v      = r2_score(obs_v, pred_v)
                da_v      = da(obs_v, pred_v)
                mda_v     = mda(obs_v, pred_v)
                maed_v    = maed(obs_v, pred_v)
                cetrend_v = ce_trend(obs_v, pred_v)

                metric_rows.append({
                    "Model"   : model_name,
                    "Dataset" : ds_name,
                    "Station" : station,
                    "MASE"    : mase_v,
                    "RMSSE"   : rmsse_v,
                    "R²"      : r2_v,
                    "DA"      : da_v,
                    "MDA"     : mda_v,
                    "MAED"    : maed_v,
                    "CETrend" : cetrend_v,
                    "Fit_s"   : round(fit_t, 2),
                    "Pred_s"  : round(pred_t, 2),
                })
                print(f"    ✓ {station}: MASE={mase_v:.4f}  RMSSE={rmsse_v:.4f}  "
                      f"R²={r2_v:.4f}  DA={da_v:.4f}  fit={fit_t:.1f}s")
            except Exception as e:
                print(f"    ✗ {station}: ERROR — {e}")
                forecasts[ds_name][model_name][station] = None

### DeepAR (neuralforecast — manual implementation)

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

from neuralforecast import NeuralForecast
from neuralforecast.models import DeepAR as NF_DeepAR

deepar_forecasts   = {}
deepar_metric_rows = []

for ds_name, series_dict in DATASETS.items():
    deepar_forecasts[ds_name] = {}

    train_frames = []
    for station in sorted(series_dict):
        y_tr = train_series[ds_name][station].ffill().bfill()
        df_s = y_tr.reset_index()
        df_s.columns = ["ds", "y"]
        df_s["unique_id"] = station
        train_frames.append(df_s)
    train_df = pd.concat(train_frames, ignore_index=True)

    print(f"  Training DeepAR | {ds_name} ...", end=" ", flush=True)

    model = NF_DeepAR(
        h=PREDICTION_LENGTH,
        input_size=24 * 7,
        lstm_hidden_size=64,
        lstm_n_layers=2,
        max_steps=50,
        scaler_type="standard",
        logger=False,
        enable_progress_bar=False,
    )
    nf = NeuralForecast(models=[model], freq="h")
    t0_fit = time.time()
    nf.fit(train_df)
    fit_t = time.time() - t0_fit
    t0_pred = time.time()
    preds_df = nf.predict()
    pred_t = time.time() - t0_pred
    n_s = len(sorted(series_dict))
    print("✓")

    for station in sorted(series_dict):
        pred_s = (
            preds_df[preds_df["unique_id"] == station]
            .set_index("ds")["DeepAR"]
        )
        pred_s.index.name = "timestamp"
        deepar_forecasts[ds_name][station] = pred_s

        y_test = test_raw_sk[station]      # ground truth always = original raw
        pred_aligned = pred_s.reindex(y_test.index)
        mask  = y_test.notna() & pred_aligned.notna()
        obs_v  = y_test[mask].values
        pred_v = pred_aligned[mask].values
        if len(obs_v):
            y_tr_v = train_series[ds_name][station].values

            mase_v    = mase(obs_v, pred_v, y_tr_v)
            rmsse_v   = rmsse(obs_v, pred_v, y_tr_v)
            r2_v      = r2_score(obs_v, pred_v)
            da_v      = da(obs_v, pred_v)
            mda_v     = mda(obs_v, pred_v)
            maed_v    = maed(obs_v, pred_v)
            cetrend_v = ce_trend(obs_v, pred_v)

            deepar_metric_rows.append({
                "Model"   : "DeepAR",
                "Dataset" : ds_name,
                "Station" : station,
                "MASE"    : mase_v,
                "RMSSE"   : rmsse_v,
                "R²"      : r2_v,
                "DA"      : da_v,
                "MDA"     : mda_v,
                "MAED"    : maed_v,
                "CETrend" : cetrend_v,
                "Fit_s"   : round(fit_t / n_s, 2),
                "Pred_s"  : round(pred_t / n_s, 2),
            })

deepar_df = pd.DataFrame(deepar_metric_rows)

print(f"\n{'═'*70}")
print("  Model: DeepAR")
print(f"{'═'*70}")

table = deepar_df.pivot_table(
    index="Station", columns="Dataset",
    values=["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], aggfunc="first",
)
table = table.reindex(columns=pd.MultiIndex.from_product(
    [["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], sorted(DATASETS.keys())]
))
table.columns = [f"{m} [{d}]" for m, d in table.columns]
table.loc["MEAN"] = table.mean()
display(table.round(4))

csv_path = os.path.join(data_dir, "sktime_metrics_DeepAR.csv")
table.round(4).to_csv(csv_path, sep=";", decimal=",")
print(f"  → Saved: {csv_path}")

## Evaluation and Comparison

In [ ]:
metrics_df = pd.DataFrame(metric_rows)

DS_LABELS = {
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}

for model_name in MODELS_CFG:
    print(f"\n{'═'*70}")
    print(f"  Model: {model_name}")
    print(f"{'═'*70}")
    sub = metrics_df[metrics_df["Model"] == model_name].copy()
    table = sub.pivot_table(
        index="Station", columns="Dataset",
        values=["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], aggfunc="first",
    )
    table = table.reindex(columns=pd.MultiIndex.from_product(
        [["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], sorted(DATASETS.keys())]
    ))
    table.columns = [f"{m} [{d}]" for m, d in table.columns]
    table.loc["MEAN"] = table.mean()
    display(table.round(4))

    csv_path = os.path.join(data_dir, f"sktime_metrics_{model_name}.csv")
    table.round(4).to_csv(csv_path, sep=";", decimal=",")
    print(f"  → Saved: {csv_path}")

In [ ]:
# ── Comparison: mean MASE and RMSSE by model and dataset ─────────────────────
_all_tmp = pd.DataFrame(metric_rows + deepar_metric_rows)

_DS_LABELS = {
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}

comparison = (
    _all_tmp
    .assign(Dataset_label=lambda d: d["Dataset"].map(_DS_LABELS))
    .groupby(["Model", "Dataset_label"])[["MASE", "RMSSE"]]
    .mean()
    .round(4)
    .unstack("Dataset_label")
    .reindex(columns=pd.MultiIndex.from_product(
        [["MASE", "RMSSE"], ["Baseline I", "Baseline II", "Hierarchical Filling"]]
    ))
    .sort_values(("MASE", "Hierarchical Filling"))
)

print("\n── Comparison: mean MASE and RMSSE by model and dataset ──")
print(comparison.to_string())

In [ ]:
import matplotlib.pyplot as plt
import os

DS_ORDER = ["baseline_I", "baseline_II", "complete_series"]
DS_LABELS_PLOT = {
    "baseline_I":      "Baseline I",
    "baseline_II":     "Baseline II",
    "complete_series": "Hierarchical Filling",
}

CONTEXT_DAYS = 30  # training days shown before the test window
plots_dir = os.path.join(data_dir, "plots_sktime")
os.makedirs(plots_dir, exist_ok=True)

for station in sorted(series_raw):
    y_test = test_raw_sk[station]

    fig, axes = plt.subplots(1, len(DS_ORDER), figsize=(18, 3), sharey=True)
    fig.suptitle(f"{station} \u2014 Jan\u2013Mar 2025  |  sktime", fontsize=11, fontweight="bold")

    for ax, ds_name in zip(axes, DS_ORDER):
        if station not in DATASETS[ds_name]:
            ax.set_visible(False)
            continue

        # Training context: last CONTEXT_DAYS before the test cutoff
        y_ctx = train_series[ds_name][station].iloc[-(24 * CONTEXT_DAYS):]
        ax.plot(y_ctx.index, y_ctx.values, color="gray", lw=1,
                alpha=0.7, label="Train (context)")

        ax.plot(y_test.index, y_test.values, color="steelblue",
                lw=1.5, label="Observed (raw)")

        for model_name in MODELS_CFG:
            pred = forecasts[ds_name][model_name].get(station)
            if pred is not None:
                ax.plot(pred.index, pred.values, lw=1,
                        linestyle="--", label=model_name)

        deepar_fc = deepar_forecasts.get(ds_name, {}).get(station)
        if deepar_fc is not None:
            fc_d = deepar_fc.reindex(y_test.index)
            ax.plot(fc_d.index, fc_d.values, lw=1,
                    linestyle="-.", label="DeepAR")

        ax.axvline(CUTOFF, color="black", lw=0.8, linestyle=":")
        ax.set_title(DS_LABELS_PLOT[ds_name], fontsize=9)
        ax.set_ylabel("DO (mg/L)" if ax is axes[0] else "")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.tick_params(axis="x", labelrotation=30, labelsize=7)

    plt.tight_layout()
    fig.savefig(
        os.path.join(plots_dir, f"{station}_forecast.png"),
        dpi=300, bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

print(f"Plots saved to: {plots_dir}")


## Final Summary Table — All Metrics × Baseline I / Baseline II / Hierarchical Filling

In [ ]:
# ── Final summary table: Baseline I / Baseline II / Hierarchical Filling ───────────
# Same structure as the AutoGluonTS final table (pivot_table + unstack).

all_metrics_df = pd.DataFrame(metric_rows + deepar_metric_rows)
metrics_order  = ["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"]

DS_LABELS = {
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}
all_metrics_df["Dataset"] = all_metrics_df["Dataset"].map(DS_LABELS).fillna(all_metrics_df["Dataset"])

dataset_order = ["Baseline I", "Baseline II", "Hierarchical Filling"]

# ── 1. Summary by model (mean over stations) ──────────────────────────────────
summary = (
    all_metrics_df
    .groupby(["Model", "Dataset"])[metrics_order]
    .mean()
    .round(4)
)
summary_pivot = summary.unstack("Dataset")
summary_pivot = summary_pivot.reindex(
    columns=pd.MultiIndex.from_product([metrics_order, dataset_order])
)
summary_pivot.columns = [f"{m} [{d}]" for m, d in summary_pivot.columns]

print("\n" + "═" * 90)
print("  FINAL TABLE — Mean by Model (all stations)")
print("═" * 90)
display(summary_pivot)

csv_final = os.path.join(data_dir, "sktime_metrics_FINAL.csv")
summary_pivot.to_csv(csv_final, sep=";", decimal=",")
print(f"\n  → Saved: {csv_final}")

# ── 2. Full table by station and model ────────────────────────────────────────
print("\n" + "═" * 90)
print("  FINAL TABLE — By Station and Model")
print("═" * 90)

full_pivot = all_metrics_df.pivot_table(
    index=["Model", "Station"],
    columns="Dataset",
    values=metrics_order,
    aggfunc="first",
).round(4)
full_pivot = full_pivot.reindex(
    columns=pd.MultiIndex.from_product([metrics_order, dataset_order])
)
full_pivot.columns = [f"{m} [{d}]" for m, d in full_pivot.columns]

display(full_pivot)

csv_full = os.path.join(data_dir, "sktime_metrics_FINAL_completa.csv")
full_pivot.to_csv(csv_full, sep=";", decimal=",")
print(f"  → Saved: {csv_full}")

In [ ]:
# ── Per-station table: Baseline I / Baseline II / Hierarchical Filling ──────────────

station_df = pd.DataFrame(metric_rows + deepar_metric_rows)

DS_LABELS = {
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}
station_df["Dataset"] = station_df["Dataset"].map(DS_LABELS).fillna(station_df["Dataset"])

metrics = ["MASE", "RMSSE", "R\u00b2", "DA", "MDA", "MAED", "CETrend"]
dataset_order = ["Baseline I", "Baseline II", "Hierarchical Filling"]

pivot = station_df.pivot_table(
    index=["Model", "Station"],
    columns="Dataset",
    values=metrics,
    aggfunc="first",
)
pivot = pivot.reindex(columns=pd.MultiIndex.from_product([metrics, dataset_order]))
pivot.columns = [f"{m} [{d}]" for m, d in pivot.columns]
pivot = pivot.reset_index()
pivot = pivot.sort_values(["Model", "Station"]).reset_index(drop=True)

bI_cols   = [c for c in pivot.columns if "[Baseline I]"      in c]
bII_cols  = [c for c in pivot.columns if "[Baseline II]"     in c]
trat_cols = [c for c in pivot.columns if "[Hierarchical Filling]"  in c]
metric_cols = bI_cols + bII_cols + trat_cols

_hb = {"R\u00b2", "DA", "MDA"}
styled_station = (
    pivot.style
    .format({c: "{:.4f}" for c in metric_cols})
    .set_caption("sktime \u2014 Metrics by Station: Baseline I / Baseline II / Hierarchical Filling")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center")]},
        {"selector": "td",      "props": [("text-align","right")]},
    ])
    .highlight_min(subset=[c for c in trat_cols if not any(m in c for m in _hb)], color="lightgreen", axis=0)
    .highlight_max(subset=[c for c in trat_cols if     any(m in c for m in _hb)], color="lightgreen", axis=0)
)
display(styled_station)

csv_station_path = os.path.join(data_dir, "sktime_metrics_por_estacao.csv")
pivot.to_csv(csv_station_path, index=False, sep=";", decimal=",")
print(f"\u2192 Source: sktime_metrics_por_estacao.csv  ({pivot.shape[0]} rows)")

In [ ]:
# ── Statistical tests: Baselines vs Hierarchical Filling \u2014 by Model \u00d7 Metric ────────
# Two paired comparisons: (Baseline I vs Treated) and (Baseline II vs Treated).
# H0: no difference. Mean \u0394 > 0 \u2192 Treated improved over the baseline.

from scipy import stats

stat_df = pd.read_csv(csv_station_path, sep=";", decimal=",")

_METRICS     = ["MASE", "RMSSE", "R\u00b2", "DA", "MDA", "MAED", "CETrend"]
_COMPARISONS = [
    ("Baseline I",  "[Baseline I]",      "[Hierarchical Filling]"),
    ("Baseline II", "[Baseline II]",     "[Hierarchical Filling]"),
]

def _run_paired(df, comparisons, metric_names):
    rows = []
    for model, grp in df.groupby("Model"):
        for comp_label, ref_tag, trat_tag in comparisons:
            for mname in metric_names:
                ref_col  = f"{mname} {ref_tag}"
                trat_col = f"{mname} {trat_tag}"
                if ref_col not in grp.columns or trat_col not in grp.columns:
                    continue
                ref_v  = grp[ref_col].values
                trat_v = grp[trat_col].values
                diff   = ref_v - trat_v

                t_stat, t_p = stats.ttest_rel(ref_v, trat_v)
                try:
                    w_stat, w_p = stats.wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
                except ValueError:
                    w_stat, w_p = np.nan, np.nan

                rows.append({
                    "Comparison" : comp_label,
                    "Model"      : model,
                    "Metric"     : mname,
                    "n"          : len(diff),
                    "Mean \u0394"     : diff.mean(),
                    "Mean \u0394 (%)" : (diff / np.abs(ref_v) * 100).mean(),
                    "t-stat"     : t_stat,
                    "t p-value"  : t_p,
                    "W-stat"     : w_stat,
                    "W p-value"  : w_p,
                    "t sig."     : "\u2713" if t_p < 0.05 else "",
                    "W sig."     : "\u2713" if (not np.isnan(w_p) and w_p < 0.05) else "",
                })
    return pd.DataFrame(rows)

tests_df = _run_paired(stat_df, _COMPARISONS, _METRICS)

def color_pval(v):
    if isinstance(v, float) and not np.isnan(v):
        if v < 0.05: return "background-color: #c6efce; color: #276221"
        if v < 0.10: return "background-color: #ffeb9c; color: #9c6500"
    return ""

def color_delta(v):
    if isinstance(v, float):
        return "color: green" if v > 0 else ("color: red" if v < 0 else "")
    return ""

dcols = ["Comparison","Model","Metric","n","Mean \u0394","Mean \u0394 (%)","t-stat","t p-value","W-stat","W p-value","t sig.","W sig."]
styled_tests = (
    tests_df[dcols].style
    .format({
        "Mean \u0394":     "{:+.4f}",
        "Mean \u0394 (%)": "{:+.2f}%",
        "t-stat":      "{:.3f}",
        "t p-value":   "{:.4f}",
        "W-stat":      lambda v: f"{v:.1f}" if not (isinstance(v, float) and np.isnan(v)) else "\u2014",
        "W p-value":   lambda v: f"{v:.4f}" if not (isinstance(v, float) and np.isnan(v)) else "\u2014",
    })
    .applymap(color_pval,  subset=["t p-value", "W p-value"])
    .applymap(color_delta, subset=["Mean \u0394", "Mean \u0394 (%)"])
    .set_caption(
        "Paired tests: Baselines vs Hierarchical Filling by Model \u00d7 Metric  "
        "(\u0394 > 0 = Treated improved | green = p < 0.05 | yellow = p < 0.10)"
    )
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center")]},
        {"selector": "td",      "props": [("text-align","right")]},
    ])
)
display(styled_tests)

csv_tests_path = os.path.join(data_dir, "sktime_testes_estatisticos.csv")
tests_df.to_csv(csv_tests_path, index=False, sep=";", decimal=",")
print(f"\u2192 Saved: {csv_tests_path}")
print()
print("Note: n=6 stations \u2192 limited power (min Wilcoxon p \u2248 0.125). Use Mean \u0394 (%) as practical effect size.")

In [ ]:
# ── Timing summary \u2014 Fit and Prediction by Model \u00d7 Dataset ───────────────────
all_t_sk = pd.DataFrame(metric_rows + deepar_metric_rows)
all_t_sk["Dataset"] = all_t_sk["Dataset"].map({
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}).fillna(all_t_sk["Dataset"])

time_summary_sk = (
    all_t_sk.groupby(["Model", "Dataset"])[["Fit_s", "Pred_s"]]
    .mean().round(2).unstack("Dataset")
)
time_summary_sk = time_summary_sk.reindex(
    columns=pd.MultiIndex.from_product(
        [["Fit_s", "Pred_s"], ["Baseline I", "Baseline II", "Hierarchical Filling"]]
    )
)
time_summary_sk.columns = [f"{m} [{d}]" for m, d in time_summary_sk.columns]
time_summary_sk.loc["MEAN"] = time_summary_sk.mean()

print("\n" + "\u2550"*80)
print("  Mean time per station \u2014 Fit (s) and Prediction (s)")
print("\u2550"*80)
display(time_summary_sk.round(2))

csv_time_sk = os.path.join(data_dir, "sktime_tempos.csv")
time_summary_sk.round(2).to_csv(csv_time_sk, sep=";", decimal=",")
print(f"  \u2192 Saved: {csv_time_sk}")